In [ ]:
#   4.1 制作数据集
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline

# 展示高清图
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats('svg')

# 准备数据集
df = pd.read_csv('Data.csv', index_col=0)   # 导入数据
arr = df.values                     # Pandas对象退化为Numpy数组
arr = arr.astypr(np.float32)        # 转为float32类型数组(适应nividia)
ts = torch.tensor(arr)              # 数组转为张量
ts = ts.to('cuda')                  # 把训练集搬到cuda上
print(ts.shape)

# 划分训练集与测试集（通用型代码！！！）
train_size = int(0.8*len(ts))     # 训练集的样本数量
test_size = len(ts) - train_size  # 测试集的样本数量
Data = ts[torch.randperm( ts.size(0)) , : ]   # 打乱样本的顺序
train_Data = Data[:train_size, :]   # 训练集样本（0~8000行，所有列）
test_Data = Data[train_size:, :]    # 测试集样本（8000~,所有列）
print(train_Data.size())
print(test_Data.size())

In [ ]:
#   4.2 搭建神经网络
class DNN(nn.Module):

    def __init__(self):
        '''搭建神经网络各层'''
        super(DNN, self).__init__()
        self.net = nn.Sequential(           # 按顺序搭建各层
            nn.Linear(8, 32), nn.Sigmoid(),
            nn.Linear(32, 8), nn.Sigmoid(),
            nn.Linear(8, 4), nn.Sigmoid(),
            nn.Linear(4, 1), nn.Sigmoid()
        )

    def forward(self, x):
        '''前向传播'''
        y = self.net(x)     # x即输入数据
        return y            # y即输出数据

model = DNN()       # 创建子类的实例
# model = DNN().to('cuda:0')    # 搬到GPU上
print(model)

In [ ]:
#   4.3 训练网络

# 损失函数的选择
loss_fn = nn.BCELoss(reduction='mean')

# 优化算法的选择
learning_rate = 0.005   # 设置学习率
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

epochs = 5000
losses = []     # 记录损失函数变化的列表

# 给训练集划分输入与输出
X = train_Data[:, :-1]                     # 前8列为输入特征
Y = train_Data[:, -1].reshape((-1,1))      # 后1列为输出特征,且升为二阶

for epoch in range(epochs):
    Pred  =model(X)             # 一次前向传播（BGD）
    loss = loss_fn(Pred, Y)     # 计算损失函数
    losses.append(loss.item())  # 记录损失函数的变化(降级为元素，加入列表)
    optimizer.zero_grad()       # 清理上一轮滞留的梯度
    loss.backward()             # 一次反向传播
    optimizer.step()            # 优化内部参数

Fig = plt.figure()
plt.plot(range(epochs), losses)
plt.ylabel('loss')
plt.xlabel('epoch')
plt.show()

In [ ]:
#   4.4 测试网络

# 给测试集划分输入输出
X = train_Data[:, :-1]                    # 前8列为输入特征
Y = train_Data[:, -1].reshape((-1,1))     # 后1列为输出特征

with torch.no_grad():       # 关闭梯度计算功能
    Pred =model(X)          # 一次前向传播(批量)
    Pred[Pred >= 0.5] = 1
    Pred[Pred < 0.5] = 0
    correct = torch.sum((Pred==Y).all(1))   # 预测正确的样本
    total = Y.size(0)                       # 全部的样本数量

print(f'测试集精准度:{100*correct/total} % ')